In [0]:
%pip install prophet
%pip install optuna

In [0]:
CATALOG_NAME = "mini_project"
SCHEMA_NAME = "gold_layer"

In [0]:
import optuna
import mlflow
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics

In [0]:
date_dim_df = spark.read.table('mini_project.gold_layer.date_dim')

display(date_dim_df)

In [0]:
sales_order_df = spark.read.table('mini_project.gold_layer.sales_order')

display(sales_order_df)

In [0]:
from pyspark.sql import functions as F

# 1) Prepare sales line amounts + normalize order date to DATE
sales_prepped_df = (
    sales_order_df
    .filter(F.col("status") == "Shipped")
    .withColumn("order_date_ts", F.to_timestamp("order_date"))
    .withColumn("order_date", F.to_date("order_date_ts"))
    .withColumn(
        "line_sales_amount",
        F.col("order_qty").cast("double")
        * F.col("unit_price").cast("double")
        * (F.lit(1.0) - F.col("unit_price_discount").cast("double"))
    )
)

# 2) Aggregate sales to daily level first
daily_sales_df = (
    sales_prepped_df
    .groupBy("order_date")
    .agg(F.sum("line_sales_amount").alias("daily_sales"))
)

# 3) Build month calendar from date_dim (one row per month)
#    ds = first day of month (Prophet-friendly)
month_calendar_df = (
    date_dim_df
    .withColumn("date", F.to_date("date"))
    .withColumn("ds", F.to_date(F.date_trunc("month", F.col("date"))))
    .select("ds", "year", "quarter", "month", "month_name")
    .dropDuplicates(["ds"])
    .orderBy("ds")
)

# 4) Join daily sales to date_dim, then roll up to month
#    This ensures months with no sales still exist
monthly_prophet_spark_df = (
    date_dim_df
    .withColumn("date", F.to_date("date"))
    .join(
        daily_sales_df,
        date_dim_df["date"] == daily_sales_df["order_date"],
        how="left"
    )
    .filter(F.col("year")!=2025)
    .withColumn("daily_sales", F.coalesce(F.col("daily_sales"), F.lit(0.0)))
    .withColumn("ds", F.to_date(F.date_trunc("month", F.col("date"))))
    .groupBy("ds")
    .agg(F.sum("daily_sales").alias("y"))
    .orderBy("ds")
)

display(monthly_prophet_spark_df)

In [0]:

with mlflow.start_run():
    # Create and fit Prophet model
    # model = Prophet(
    #     changepoint_prior_scale=0.05,
    #     seasonality_prior_scale=10,
    #     yearly_seasonality=True,
    #     weekly_seasonality=True,
    # )
        
    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False
    )

    model.fit(monthly_prophet_spark_df.toPandas())

    # Log model parameters
    # mlflow.log_params(
    #     {
    #         "changepoint_prior_scale": 0.05,
    #         "seasonality_prior_scale": 10,
    #         "yearly_seasonality": True,
    #         "weekly_seasonality": True,
    #     }
    # )

    mlflow.log_params(
        {
            "yearly_seasonality" : True,
            "weekly_seasonality" : False,
            "daily_seasonality" : False
        }
    )
    # # Cross-validation
    # cv_results = cross_validation(
    #     model,
    #     initial="730 days",
    #     period="180 days",
    #     horizon="365 days",
    # )

    # # Log performance metrics
    # metrics = performance_metrics(cv_results)
    # mlflow.log_metrics(metrics[["mse", "rmse", "mae", "mape"]].mean().to_dict())

    # # Log model
    mlflow.prophet.log_model(
        pr_model=model, input_example=monthly_prophet_spark_df.toPandas().head(),        artifact_path="prophet-gross-sales-model",
        registered_model_name=f"{CATALOG_NAME}.{SCHEMA_NAME}.test-gross-sales-model"
    )

In [0]:
import optuna

# ADD THIS (before objective)
best_model_holder = {"model": None, "mape": float("inf")}

# ADD THIS (fixed Prophet params you always want)
forced_prophet_params = {
    "yearly_seasonality": True,
    "weekly_seasonality": False,
    "daily_seasonality": False,
}


def objective(trial, df):
    """Optuna objective for Prophet hyperparameter tuning."""

    with mlflow.start_run(nested=True):
        # Define hyperparameter search space
        params = {
            "changepoint_prior_scale": trial.suggest_float(
                "changepoint_prior_scale", 0.001, 0.5
            ),
            "seasonality_prior_scale": trial.suggest_float(
                "seasonality_prior_scale", 0.01, 10
            ),
            "holidays_prior_scale": trial.suggest_float(
                "holidays_prior_scale", 0.01, 10
            ),
            "seasonality_mode": trial.suggest_categorical(
                "seasonality_mode", ["additive", "multiplicative"]
            ),
        }

        # Train model
        model = Prophet(**params, **forced_prophet_params)
        model.fit(df)

        # Cross-validation
        cv_results = cross_validation(
            model, initial="366 days", period="90 days", horizon="180 days"
        )
        metrics = performance_metrics(cv_results)
        mape = metrics["mape"].mean()

        # Log parameters and metrics
        mlflow.log_params(params)
        mlflow.log_metric("mape", mape)

        # Optional: log forced params too so they show in MLflow runs
        mlflow.log_params(forced_prophet_params)

        # ADD THIS (capture best fitted model)
        if mape < best_model_holder["mape"]:
            best_model_holder["mape"] = mape
            best_model_holder["model"] = model

        return mape


# Run optimization
with mlflow.start_run(run_name="Prophet HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial, monthly_prophet_spark_df.toPandas()), n_trials=50)

     # Log best tuned parameters + metric
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    mlflow.log_metric("best_mape", study.best_value)

    # Log forced params at parent level too (optional but useful)
    mlflow.log_params({f"best_{k}": v for k, v in forced_prophet_params.items()})

    # NEW: retrain final/best model using best hyperparameters found
    best_params = {**study.best_params, **forced_prophet_params}
    best_model = Prophet(**best_params)
    best_model.fit(monthly_prophet_spark_df.toPandas())

    # NEW: log the retrained best model
    mlflow.prophet.log_model(
        pr_model=model, input_example=monthly_prophet_spark_df.toPandas().head(),        artifact_path="prophet-gross-sales-model",
        registered_model_name=f"{CATALOG_NAME}.{SCHEMA_NAME}.gross-sales-model"
    )

    print("Best trial params:", study.best_params)
    print("Forced params:", forced_prophet_params)
    print("Final model trained with:", best_params)

In [0]:
monthly_prophet_spark_df.toPandas()

In [0]:
from prophet import Prophet

model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False
)

model.fit(monthly_prophet_spark_df.toPandas())


In [0]:
future_pd = model.make_future_dataframe(
    periods=12,
    freq="MS",
    include_history=True
)

# predict over the dataset
forecast_pd = model.predict(future_pd)

In [0]:
fig = model.plot(forecast_pd)
fig2 = model.plot_components(forecast_pd)